# 11 · Participação: abstenção, brancos e nulos

**Denominadores** (explícitos nos nomes das colunas):
- **abstenção** = abstenção ÷ **eleitorado**;
- **brancos** e **nulos** = ÷ **comparecimento**;
- `nulos_tvn` = nulos comuns + nulos técnicos (`vn + vnt`); `nulos_vn` = só nulos comuns. A diferença é `vnt` (nulos técnicos): existe em 2026 e é zero em 2022.

**Ressalva ecológica**: as correlações são entre **médias de município**. Não dizem nada sobre o comportamento de um eleitor.

**Corte de eleitorado no ranking**: a abstenção de municípios pequenos é instável; o top/bottom usa só municípios acima de um corte, justificado pela sensibilidade da lista (seção 3).

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
b = cp.base_2026()
bb = pa.adicionar_abstencao(base.apenas_brasil(b))
print(len(bb), "municípios BR")

## 1. Por região, porte e UF

In [ ]:
print(base.participacao_agregada(bb, ["regiao"]).round(2).to_string(index=False))
print(
    base.participacao_agregada(
        bb.dropna(subset=["faixa"]).assign(faixa=lambda d: d.faixa.astype(str)), ["faixa"]
    )
    .round(2)
    .to_string(index=False)
)
uf_p = base.participacao_agregada(bb, ["uf"])
print(
    uf_p[
        [
            "uf",
            "pct_abstencao_sobre_eleitorado",
            "pct_brancos_sobre_comparecimento",
            "pct_nulos_tvn_sobre_comparecimento",
            "pct_nulos_vn_sobre_comparecimento",
        ]
    ]
    .round(2)
    .sort_values("pct_abstencao_sobre_eleitorado", ascending=False)
    .to_string(index=False)
)

## 2. Correlação ecológica com o voto facultativo (16–17 e 70+)

In [ ]:
bb2 = bb.assign(
    pct_brancos=100 * bb.brancos / bb.comparecimento,
    pct_nulos=100 * bb.nulos_tvn / bb.comparecimento,
)
pares = [
    ("pct_faixa_facultativa_16_17", "pct_abstencao_sobre_eleitorado", "abstenção"),
    ("pct_faixa_facultativa_70_mais", "pct_abstencao_sobre_eleitorado", "abstenção"),
    ("pct_faixa_facultativa_16_17", "pct_brancos", "brancos"),
    ("pct_faixa_facultativa_70_mais", "pct_nulos", "nulos (tvn)"),
]
rows = []
for x, y, rot in pares:
    s = pa.correlacao_ecologica(bb2, x, y, "spearman")
    rows.append(
        {"x": x, "y": rot, "rho_spearman": round(s["r"], 3), "p_valor": s["p_valor"], "n": s["n"]}
    )
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for a, (x, lab) in zip(
    ax,
    [
        ("pct_faixa_facultativa_16_17", "% eleitorado 16–17 anos"),
        ("pct_faixa_facultativa_70_mais", "% eleitorado 70+ anos"),
    ],
):
    d = bb2[[x, "pct_abstencao_sobre_eleitorado"]].dropna()
    a.scatter(
        d[x],
        d["pct_abstencao_sobre_eleitorado"],
        s=5,
        alpha=0.25,
        color="#555555",
        edgecolors="none",
    )
    a.set_xlabel(lab)
    a.set_ylabel("% abstenção (sobre eleitorado)")
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle(
    "Correlação ecológica (município): facultativo × abstenção. NÃO é comportamento individual",
    x=0.01,
    ha="left",
)
g.salvar(fig, "participacao_correlacao_facultativa")
plt.show()

## 3. Sensibilidade ao corte de eleitorado

In [ ]:
cortes = [0, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000]
est = pa.estabilidade_cortes(bb, cortes, n=20)
ref = est[cortes.index(10_000)]
rows = []
for e in est:
    rows.append(
        {
            "corte": e.corte,
            "municipios_elegiveis": e.n_municipios,
            "top20_jaccard_vs_10k": round(pa.jaccard(e.top_n, ref.top_n), 2),
            "bottom20_jaccard_vs_10k": round(pa.jaccard(e.bottom_n, ref.bottom_n), 2),
            "frac_top20_menor_50k": round(e.fracao_menores_50k_no_top, 2),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

Corte escolhido: **10.000 eleitores** (perto da mediana dos municípios, 9.279). Abaixo dele, o top-20 muda quase por completo entre cortes vizinhos, o que indica ruído de município pequeno. Registrado em `docs/DECISOES.md` (D-021).

## 4. Top e bottom 20 por abstenção (corte 10.000)

In [ ]:
r = pa.ranking_participacao(bb, corte_eleitorado=10_000, n=20)
print("MAIOR abstenção")
print(r["maior_abstencao"].round(2).to_string(index=False))
print("MENOR abstenção")
print(r["menor_abstencao"].round(2).to_string(index=False))